## Lab 4: Markov Chain → Q-Learning

### Markov Chain
- Shift from one state to another
- Depends only on the next step, not the sequence of past steps (memoryless)

**Properties:**
1. State space
2. Transition Probability Matrix
3. Stationary distribution → no change in probability distribution

### Markov Decision Process (MDP)
Sequential decision-making problem where the outcome of an action is uncertain.
- State (S), Action (A), Transition (P)
- Reward function (R), γ (discount rate)

### Q-Learning
- Trial-and-error algorithm
- Learns the long-term value of a specific action in a specific situation
- Q → "Quality": a lookup table (Q-table) — if state 'S' and take action 'A',
  how good will my total future payoff be?

**Mathematical Equation:**

Q(s, a) ← Q(s, a) + α [ R + γ · maxₐ' Q(s', a') − Q(s, a) ]

Where:
- **Q(s, a)** (old value) → the agent's current estimate of how good action *a* is in state *s*
- **α** (learning rate) → between 0 and 1 → (0 = learn nothing, 1 = fully learn the new value)
- **R** (immediate reward) → penalty/reward score
- **γ** (discount factor) → between 0 and 1 → weighs future reward against the immediate one
- **s'** → next state; maxₐ' Q(s', a') → best expected future value

### Example: 4×4 Grid World
S0 S1 S2 S3
S4 S5* S6 S7
S8 S9 S10* S11
S12 S13 S14 S15(Goal)
- Start state: S0
- Goal state: S15 → R = +10
- Obstacles: S5, S10 → R = −10 (episode ends)
- Normal step → R = −1
- Actions: ↑ (Up), ↓ (Down), ← (Left), → (Right)
- α = 0.5, γ = 0.9

**Episode 1:**
- S0 → S1: Q(S0, →) ← 0 + 0.5[−1 + 0.9(0) − 0] = −0.5
- S1 → S5 (obstacle): Q(S1, ↓) ← 0 + 0.5[−10 + 0.9(0) − 0] = −5

**Episode 2:**
- S0 → S4: Q(S0, ↓) = −0.5
- ...
- S14 → S15 (goal): Q(S14, →) = 0 + 0.5[10 + 0.9(0)] = 5

**Episode 3 (back-propagation):**
- Q(S14, →) = +5 (carried over)
- Q(S13, →) ← −0.5 + 0.5[−1 + 0.9(5) − (−0.5)] = −0.5? 


In [2]:
"""
Q-Learning on a 4x4 Grid World (Markov Decision Process)
----------------------------------------------------------
Copy this cell into a Jupyter notebook and run.

Grid layout (states S0..S15):
    S0   S1   S2   S3
    S4   S5*  S6   S7
    S8   S9   S10* S11
    S12  S13  S14  S15 (Goal)

    * = obstacle (episode ends, reward -10)
    Goal (S15): reward +10, episode ends
    Every other move: reward -1

Q-learning update rule:
    Q(s,a) <- Q(s,a) + alpha * [ R + gamma * max_a' Q(s',a') - Q(s,a) ]

For every episode, the full Q-table (16 states x 4 actions) is printed
as a table, along with the path the agent took.
"""

import random
import pandas as pd

# ---------------------------------------------------------
# Environment setup
# ---------------------------------------------------------
GRID_SIZE = 4
NUM_STATES = GRID_SIZE * GRID_SIZE     # S0 ... S15
GOAL_STATE = 15
OBSTACLES = {5, 10}
START_STATE = 0

ACTIONS = ["Up", "Down", "Left", "Right"]

ALPHA = 0.5      # learning rate
GAMMA = 0.9      # discount factor
EPSILON = 0.2    # exploration rate (epsilon-greedy)
NUM_EPISODES = 5
MAX_STEPS_PER_EPISODE = 50
RANDOM_SEED = 42


def next_state(state, action):
    """Return the resulting state after taking `action` from `state`.
    Moving off the grid keeps the agent in place."""
    row, col = divmod(state, GRID_SIZE)

    if action == "Up":
        row = max(row - 1, 0)
    elif action == "Down":
        row = min(row + 1, GRID_SIZE - 1)
    elif action == "Left":
        col = max(col - 1, 0)
    elif action == "Right":
        col = min(col + 1, GRID_SIZE - 1)

    return row * GRID_SIZE + col


def get_reward(state):
    """Reward for landing on `state`."""
    if state == GOAL_STATE:
        return 10, True          # reward, done
    elif state in OBSTACLES:
        return -10, True         # reward, done
    else:
        return -1, False         # reward, done


def q_table_to_dataframe(Q):
    """Convert the Q-table dict into a readable DataFrame: rows=states, cols=actions."""
    data = []
    for s in range(NUM_STATES):
        row = {"State": f"S{s}"}
        for a in ACTIONS:
            row[a] = round(Q[s][a], 3)
        data.append(row)
    return pd.DataFrame(data)


def choose_action(Q, state, epsilon, rng):
    """Epsilon-greedy action selection."""
    if rng.random() < epsilon:
        return rng.choice(ACTIONS)
    # exploit: pick the action with the highest Q value (break ties randomly)
    max_q = max(Q[state].values())
    best_actions = [a for a, v in Q[state].items() if v == max_q]
    return rng.choice(best_actions)


# ---------------------------------------------------------
# Initialize Q-table: all zeros
# ---------------------------------------------------------
Q = {s: {a: 0.0 for a in ACTIONS} for s in range(NUM_STATES)}

rng = random.Random(RANDOM_SEED)

print("=== Initial Step: Q-table ===")
display(q_table_to_dataframe(Q)) if "display" in dir(__builtins__) else print(q_table_to_dataframe(Q).to_string(index=False))

# ---------------------------------------------------------
# Run Q-learning episodes
# ---------------------------------------------------------
for ep in range(1, NUM_EPISODES + 1):
    state = START_STATE
    path = [f"S{state}"]
    done = False
    steps = 0

    while not done and steps < MAX_STEPS_PER_EPISODE:
        action = choose_action(Q, state, EPSILON, rng)
        s_next = next_state(state, action)
        reward, done = get_reward(s_next)

        # Q-learning update
        best_next_q = max(Q[s_next].values())
        Q[state][action] += ALPHA * (reward + GAMMA * best_next_q - Q[state][action])

        state = s_next
        path.append(f"S{state}")
        steps += 1

    print(f"\n=== Episode {ep} ===")
    print("Path taken:", " -> ".join(path))
    print(f"Ended at {path[-1]} after {steps} steps.")

    print(f"\nQ-table after Episode {ep}:")
    q_df = q_table_to_dataframe(Q)
    display(q_df) if "display" in dir(__builtins__) else print(q_df.to_string(index=False))

=== Initial Step: Q-table ===


,State,Up,Down,Left,Right
0,S0,0.0,0.0,0.0,0.0
1,S1,0.0,0.0,0.0,0.0
2,S2,0.0,0.0,0.0,0.0
3,S3,0.0,0.0,0.0,0.0
4,S4,0.0,0.0,0.0,0.0
5,S5,0.0,0.0,0.0,0.0
6,S6,0.0,0.0,0.0,0.0
7,S7,0.0,0.0,0.0,0.0
8,S8,0.0,0.0,0.0,0.0
9,S9,0.0,0.0,0.0,0.0



=== Episode 1 ===
Path taken: S0 -> S0 -> S4 -> S0 -> S0 -> S1 -> S5
Ended at S5 after 6 steps.

Q-table after Episode 1:


,State,Up,Down,Left,Right
0,S0,-0.5,-0.5,-0.5,-0.5
1,S1,0.0,-5.0,0.0,0.0
2,S2,0.0,0.0,0.0,0.0
3,S3,0.0,0.0,0.0,0.0
4,S4,-0.5,0.0,0.0,0.0
5,S5,0.0,0.0,0.0,0.0
6,S6,0.0,0.0,0.0,0.0
7,S7,0.0,0.0,0.0,0.0
8,S8,0.0,0.0,0.0,0.0
9,S9,0.0,0.0,0.0,0.0



=== Episode 2 ===
Path taken: S0 -> S0 -> S1 -> S2 -> S3 -> S3 -> S7 -> S6 -> S10
Ended at S10 after 8 steps.

Q-table after Episode 2:


...

,State,Up,Down,Left,Right
0,S0,-0.975,-0.75,-0.975,-0.875
1,S1,0.000,-5.00,-0.725,-0.500
2,S2,0.000,0.00,0.000,-0.500
3,S3,-0.500,-0.50,0.000,0.000
4,S4,-0.500,0.00,0.000,-5.000
5,S5,0.000,0.00,0.000,0.000
6,S6,0.000,-5.00,0.000,0.000
7,S7,0.000,0.00,-0.500,0.000
8,S8,0.000,0.00,0.000,0.000
9,S9,0.000,0.00,0.000,0.000



=== Episode 4 ===
Path taken: S0 -> S1 -> S0 -> S4 -> S8 -> S12 -> S8 -> S4 -> S4 -> S8 -> S8 -> S9 -> S5
Ended at S5 after 12 steps.

Q-table after Episode 4:


,State,Up,Down,Left,Right
0,S0,-0.975,-0.875,-0.975,-0.938
1,S1,0.000,-5.000,-1.200,-0.500
2,S2,0.000,0.000,0.000,-0.500
3,S3,-0.500,-0.500,0.000,0.000
4,S4,-0.500,-0.750,-0.500,-5.000
5,S5,0.000,0.000,0.000,0.000
6,S6,0.000,-5.000,0.000,0.000
7,S7,0.000,0.000,-0.500,0.000
8,S8,-0.500,-0.500,-0.500,-0.500
9,S9,-5.000,0.000,0.000,0.000



=== Episode 5 ===
Path taken: S0 -> S4 -> S0 -> S1 -> S1 -> S1 -> S2 -> S6 -> S2 -> S1 -> S2 -> S2 -> S6 -> S7 -> S11 -> S11 -> S7 -> S3 -> S2 -> S6 -> S5
Ended at S5 after 20 steps.

Q-table after Episode 5:


,State,Up,Down,Left,Right
0,S0,-0.975,-1.163,-0.975,-0.969
1,S1,-0.975,-5.000,-1.200,-0.875
2,S2,-0.500,-0.875,-0.838,-0.500
3,S3,-0.500,-0.500,-0.725,0.000
4,S4,-1.172,-0.750,-0.500,-5.000
5,S5,0.000,0.000,0.000,0.000
6,S6,-0.500,-5.000,-5.000,-0.500
7,S7,-0.500,-0.500,-0.500,0.000
8,S8,-0.500,-0.500,-0.500,-0.500
9,S9,-5.000,0.000,0.000,0.000
